In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv(r"building data/2zone_Supermarket.csv", encoding="gbk")
df.rename(columns={'Environment:Site Outdoor Air Drybulb Temperature [C](TimeStep)': 'outdoor temperature'}, inplace=True)
df

,Date/Time,outdoor temperature,BACKROOM:Zone Total Internal Total Heating Rate [W](TimeStep),SALESFLOOR:Zone Total Internal Total Heating Rate [W](TimeStep),BACKROOM LIGHTS 1:Lights Electricity Rate [W](TimeStep),SALESFLOOR LIGHTS 1:Lights Electricity Rate [W](TimeStep),BACKROOM ELECEQ 1:Electric Equipment Electricity Rate [W](TimeStep),SALESFLOOR ELECEQ 1:Electric Equipment Electricity Rate [W](TimeStep),SALESFLOOR ELECEQ 2:Electric Equipment Electricity Rate [W](TimeStep),SALESFLOOR ELECEQ 3:Electric Equipment Electricity Rate [W](TimeStep),BACKROOM:Zone Windows Total Heat Gain Rate [W](TimeStep),SALESFLOOR:Zone Windows Total Heat Gain Rate [W](TimeStep),BACKROOM:Zone Air Temperature [C](TimeStep),BACKROOM:Zone Air System Sensible Heating Rate [W](TimeStep),BACKROOM:Zone Air System Sensible Cooling Rate [W](TimeStep),SALESFLOOR:Zone Air Temperature [C](TimeStep),SALESFLOOR:Zone Air System Sensible Heating Rate [W](TimeStep),SALESFLOOR:Zone Air System Sensible Cooling Rate [W](TimeStep)
0,01/01 00:15:00,0.75,10.5,9457.5,3.0,9000.0,7.5,0.0,0.0,300.0,0.0,0.0,19.998740,892.817384,0.0,21.982192,6691.603054,0.0
1,01/01 00:30:00,1.50,10.5,9457.5,3.0,9000.0,7.5,0.0,0.0,300.0,0.0,0.0,20.013591,804.598616,0.0,21.996581,5018.467095,0.0
2,01/01 00:45:00,2.25,10.5,9457.5,3.0,9000.0,7.5,0.0,0.0,300.0,0.0,0.0,20.022188,862.534193,0.0,21.999341,6149.798386,0.0
3,01/01 01:00:00,3.00,10.5,9457.5,3.0,9000.0,7.5,0.0,0.0,300.0,0.0,0.0,20.030167,870.297423,0.0,21.999872,6336.278048,0.0
4,01/01 01:15:00,2.00,10.5,9457.5,3.0,9000.0,7.5,0.0,0.0,300.0,0.0,0.0,20.037435,880.578178,0.0,21.999975,6569.112118,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35035,12/31 23:00:00,0.00,10.5,9457.5,3.0,9000.0,7.5,0.0,0.0,300.0,0.0,0.0,19.692433,832.741763,0.0,21.972241,4008.808827,0.0
35036,12/31 23:15:00,0.00,10.5,9457.5,3.0,9000.0,7.5,0.0,0.0,300.0,0.0,0.0,19.680000,813.834794,0.0,21.902768,3945.687681,0.0
35037,12/31 23:30:00,0.00,10.5,9457.5,3.0,9000.0,7.5,0.0,0.0,300.0,0.0,0.0,19.670310,798.329552,0.0,21.840934,3921.633981,0.0
35038,12/31 23:45:00,0.00,10.5,9457.5,3.0,9000.0,7.5,0.0,0.0,300.0,0.0,0.0,19.662201,784.411293,0.0,21.783584,3912.588600,0.0


In [4]:
block_size = 4
df = df.reset_index(drop=True)
numeric_cols = df.select_dtypes(include='number').columns
block_mean = df[numeric_cols].groupby(df.index // block_size).mean()

In [8]:
selected_df = pd.DataFrame({'outdoor temperature': block_mean['outdoor temperature']})

col_to_sum = ['BACKROOM LIGHTS 1:Lights Electricity Rate [W](TimeStep)', 'SALESFLOOR LIGHTS 1:Lights Electricity Rate [W](TimeStep)', 
              'BACKROOM ELECEQ 1:Electric Equipment Electricity Rate [W](TimeStep)', 'SALESFLOOR ELECEQ 1:Electric Equipment Electricity Rate [W](TimeStep)', 
              'SALESFLOOR ELECEQ 2:Electric Equipment Electricity Rate [W](TimeStep)', 'SALESFLOOR ELECEQ 3:Electric Equipment Electricity Rate [W](TimeStep)']
selected_df['load'] = block_mean[col_to_sum].sum(axis=1) * 4
selected_df = selected_df.reset_index(drop=True)
n = len(selected_df)
selected_df['date'] = selected_df.index // 24
selected_df['hour'] = selected_df.index % 24

selected_df['date'] = np.sin(selected_df['date'] * 2 * np.pi / 365)
selected_df['hour'] = np.sin(selected_df['hour'] * 2 * np.pi / 24)

selected_df

,outdoor temperature,load,date,hour
0,1.875,37242.0,0.000000,0.000000
1,0.500,37242.0,0.000000,0.258819
2,-2.250,37242.0,0.000000,0.500000
3,-2.375,37242.0,0.000000,0.707107
4,-3.875,115248.0,0.000000,0.866025
...,...,...,...,...
8755,0.000,181290.0,-0.017213,-0.965926
8756,0.000,181290.0,-0.017213,-0.866025
8757,0.000,55248.0,-0.017213,-0.707107
8758,0.000,37242.0,-0.017213,-0.500000


In [9]:
selected_df.to_csv(r'building data/2zonesupermarket.csv', index=False)